## 한국어 뉴스 주제 분류 파인튜닝

기본 모델: `klue/bert-base`  
데이터셋: `klue/ynat`  
결과: 한국어 뉴스 제목을 주제별로 분류

파인튜닝 전에는 기본 모델이 뉴스 주제 분류용으로 학습되어 있지 않기 때문에 원하는 라벨을 제대로 예측하지 못한다.  
파인튜닝 후에는 한국어 뉴스 제목을 입력했을 때 정치, 경제, 사회, 생활문화, 세계, IT과학, 스포츠 중 하나로 분류할 수 있다.

영화리뷰 감정분석
기본 모델: beomi/kcbert-base
데이터셋: NSMC
분류 라벨: 부정, 긍정
결과: 한국어 영화 리뷰의 감정을 분류

In [ ]:
!pip install -q -U "datasets<4.0.0" transformers evaluate accelerate

In [ ]:
import torch
import numpy as np
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
import evaluate

print("GPU 사용 가능 여부:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU 이름:", torch.cuda.get_device_name(0))

데이터 불러오기

In [ ]:
from datasets import load_dataset

dataset = load_dataset("klue/klue", "ynat")

print(dataset)
print(dataset["train"][0])

In [ ]:
label_names = dataset["train"].features["label"].names
print(label_names)

기본 모델과 토크나이저 불러오기

In [ ]:
model_name = "klue/bert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)

토큰화

In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["title"],
        truncation=True,
        padding="max_length",
        max_length=64
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

평가지표 정의

 학습 중 모델 성능을 측정할 지표(정확도, F1 점수)를 설정합니다


In [ ]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=predictions, references=labels)
    f1_score = f1.compute(predictions=predictions, references=labels, average="macro")
    return {
        "accuracy": acc["accuracy"],
        "f1": f1_score["f1"]
    }

학습/검증 데이터 분리

In [ ]:
train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(3000))
eval_dataset = tokenized_dataset["validation"].shuffle(seed=42).select(range(1000))

print("학습 데이터 개수:", len(train_dataset))
print("검증 데이터 개수:", len(eval_dataset))

학습 설정 (TrainingArguments)

학습률, 배치 크기, 에폭 수 등 파인튜닝에 필요한 하이퍼파라미터를 지정합니다.



In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    report_to="none"
)

In [ ]:
from accelerate.state import AcceleratorState
AcceleratorState._reset_state()

In [ ]:
results = trainer.evaluate()
print(results)

파인튜닝 전 예측 확인

학습을 시작하기 전, 기본 모델이 뉴스 제목을 제대로 분류하지 못하는지 먼저 확인합니다.



In [ ]:
before_classifier = pipeline(
    "text-classification",
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

sample_text = "손흥민 이번 시즌 최다 득점 기록 경신"
print(before_classifier(sample_text))

모델 학습 (Trainer 실행)

Trainer에 모델, 학습 설정, 데이터, 평가지표를 연결하고 실제 파인튜닝을 진행합니다.



In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

results = trainer.evaluate()

print(f"정확도: {results['eval_accuracy'] * 100:.2f}%")
print(f"F1 점수: {results['eval_f1'] * 100:.2f}%")

파인튜닝 후 예측 확인

학습이 끝난 모델로 같은 문장들을 다시 분류해서, 파인튜닝 전과 결과를 비교합니다.



In [ ]:
after_classifier = pipeline(
    "text-classification",
    model=trainer.model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1
)

sample_texts = [
    "손흥민 이번 시즌 최다 득점 기록 경신",
    "정부, 내년도 예산안 발표",
    "새로운 스마트폰 출시로 시장 반응 뜨거워"
]

for text in sample_texts:
    result = after_classifier(text)
    print(f"문장: {text}")
    print(f"결과: {result[0]}")
    print()